# AIPerf Use Case 4 — Goodput: Measuring SLA Compliance

Raw throughput counts every completed request; **goodput** counts only the requests that met your service-level objectives (SLOs). It answers "what fraction of my users are having a good time?" — the gist's demo served 26.7 req/s but only 7.43 req/s (28%) met its SLOs.

Based on the [AIPerf office-hours gist](https://gist.github.com/BenHamm/31c648f7d7331c94c1f3a45859db6677); notes in `docs/reference/aiperf-office-hours.md`. This repo's metrics docs (`docs/metrics/`) name goodput the most decision-relevant number for model selection and sizing.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — workload and SLOs](#3-input--workload-and-slos)
4. [Test run](#4-test-run)
5. [Results analysis](#5-results-analysis)

## 1. Overview

Goodput = requests/sec meeting **all** specified SLOs simultaneously. A request that streams its first token fast but finishes late fails; so does one that finishes fast after a slow first token. Averages and single percentiles hide this because a request can look fine on each metric's marginal distribution while failing the joint requirement.

AIPerf computes it natively with one extra flag:

```
--goodput "time_to_first_token:370 request_latency:648"
```

(values in ms; metrics are the same names as in the export).

## 2. Requirements

- `aiperf` CLI installed and on `PATH` (from the NGC AIPerf image, or `pip install aiperf`). The office-hours gist pinned `release/0.3.0`; pin whatever version you use (repo convention: record the AIPerf version per run).
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- Tip: AIPerf's live dashboard is designed for a terminal. If it renders poorly inside Jupyter, check `aiperf profile --help` for a plain/simple UI option in your version.
- A workload file or synthetic settings (this notebook defaults to synthetic; point it at the Mooncake slice from the Use Case 3 notebook for the gist's exact setup).

In [ ]:
import json
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

aiperf_path = shutil.which("aiperf")
if aiperf_path is None:
    print("aiperf CLI not found on PATH — install it before running the Test run section.")
else:
    print(f"aiperf found at: {aiperf_path}")
    version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
    print((version.stdout or version.stderr).strip())


## 3. Input — workload and SLOs

The workload is whatever you'd benchmark anyway; goodput just adds pass/fail thresholds on top. Choosing thresholds is a product decision — the gist sketches three tiers:

| Tier | TTFT SLO | Request latency SLO | Typical use |
|---|---|---|---|
| Strict (premium) | 250 ms | 500 ms | interactive chat |
| Balanced (standard) | 370 ms | 648 ms | default UX target |
| Relaxed (batch) | 600 ms | 2500 ms | background generation |

If goodput comes out embarrassingly low, either the deployment is under-provisioned **or the SLOs are unrealistic for the workload** — the gist's 28% on an 8×H200 endpoint was mostly the latter (its balanced SLOs were set at the *median* of a previous run, so ~half of requests fail by construction).

## 4. Test run

In [ ]:
def run_aiperf(cmd):
    """Print and run an aiperf command from the repo root, streaming output into the notebook."""
    print("$ " + " \\\n    ".join(cmd))
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
    print(f"\nexit code: {result.returncode}")
    return result

# ---- Required ----------------------------------------------------------
URL = ""
MODEL = ""       # leave empty to auto-detect from {URL}/v1/models; set to override

# ---- SLOs (ms) ----------------------------------------------------------
TTFT_SLO = 370
LATENCY_SLO = 648

# ---- Workload: synthetic by default; set INPUT_FILE to a mooncake-format
# ---- JSONL (e.g. the Use Case 3 slice) to reproduce the gist exactly.
INPUT_FILE = ""            # e.g. "notebooks/data/mooncake_trace_5min_5x.jsonl"
CONCURRENCY = 10
REQUEST_COUNT = 100
TOKENIZER = ""
# Relative to REPO_ROOT (aiperf runs with cwd=REPO_ROOT) — keep artifacts under notebooks/.
OUTPUT_DIR = "notebooks/artifacts/aiperf-uc4-goodput"

assert URL, "Set URL above."

# Ask the endpoint what it serves — verifies the URL is reachable and spares
# the user a second copy-paste. Multi-model servers: first model wins.
if not MODEL:
    models_url = URL.rstrip("/") + "/v1/models"
    with urllib.request.urlopen(models_url, timeout=10) as resp:
        served = json.load(resp)["data"]
    assert served, f"{models_url} is reachable but lists no models."
    if len(served) > 1:
        print(f"{models_url} lists {len(served)} models: {[m['id'] for m in served]}")
    MODEL = served[0]["id"]
print(f"URL   : {URL}")
print(f"MODEL : {MODEL}")

cmd = [
    "aiperf", "profile",
    "--model", MODEL, "--url", URL,
    "--endpoint-type", "chat", "--streaming",
    "--tokenizer", TOKENIZER or MODEL,
    "--artifact-dir", OUTPUT_DIR,
    "--goodput", f"time_to_first_token:{TTFT_SLO} request_latency:{LATENCY_SLO}",
]
if INPUT_FILE:
    cmd += ["--input-file", INPUT_FILE, "--custom-dataset-type", "mooncake_trace", "--fixed-schedule"]
else:
    cmd += ["--concurrency", str(CONCURRENCY), "--request-count", str(REQUEST_COUNT),
            # NOTE: --synthetic-input-tokens-mean / --output-tokens-mean are the canonical
            # long names for the gist's --isl / --osl shorthands. Used here because they are
            # guaranteed across AIPerf versions and match this repo's scenario scripts
            # (e.g. --output-tokens-mean in model-selection/scripts/run_content_generation.sh).
            "--synthetic-input-tokens-mean", "1000", "--output-tokens-mean", "500"]
run_aiperf(cmd)


## 5. Results analysis

The export's **run-totals section** now contains a **Goodput** row next to Request Throughput. Their ratio is the fraction of requests meeting all SLOs. (The CSV holds blank-line-separated sections: per-request statistics, single-value run totals, and optionally GPU telemetry — the loader below splits them.)

In [ ]:
from io import StringIO

import pandas as pd

artifact_dir = REPO_ROOT / OUTPUT_DIR
csv_path = next(artifact_dir.rglob("profile_export_aiperf.csv"), None)
assert csv_path is not None, f"No profile_export_aiperf.csv under {artifact_dir} — did the Test run complete?"


def read_export(path):
    """Split the multi-section AIPerf CSV into (stats, totals) DataFrames.

    Section 1: per-request statistics (Metric, avg, min, max, p50, ..., std).
    Section 2: single-value run totals (Metric, Value).
    Section 3 (GPU telemetry, if present) is ignored here.
    """
    sections = Path(path).read_text().strip().split("\n\n")
    stats = pd.read_csv(StringIO(sections[0]))
    totals = (pd.read_csv(StringIO(sections[1]))
              if len(sections) > 1 else pd.DataFrame(columns=["Metric", "Value"]))
    return stats, totals


stats, totals = read_export(csv_path)
pd.set_option("display.max_rows", None)

# Section 1: per-request statistics — one row per metric, columns are
# avg/min/max/percentiles/std across all requests in the run.
print("Per-request statistics (one row per metric, columns = avg/min/max/percentiles/std):")
display(stats)

# Section 2: run totals — single-value rows, including the Goodput row
# alongside Request Throughput (see the ratio computed in the next cell).
print("Run totals (single-value rows: duration, request count, throughput, goodput):")
totals


In [ ]:
def find_value(name_contains):
    """Look up a metric by name — run totals first, then the statistics section's avg."""
    m = totals[totals["Metric"].str.lower().str.contains(name_contains)]
    if not m.empty:
        return float(m["Value"].iloc[0])
    m = stats[stats["Metric"].str.lower().str.contains(name_contains)]
    return float(m["avg"].iloc[0]) if not m.empty else None


goodput = find_value("goodput")
throughput = find_value("request throughput")
if goodput is not None and throughput:
    print(f"Request throughput: {throughput:.2f} req/s")
    print(f"Goodput:            {goodput:.2f} req/s")
    print(f"=> {100 * goodput / throughput:.0f}% of requests met ALL SLOs "
          f"(TTFT <= {TTFT_SLO} ms AND latency <= {LATENCY_SLO} ms)")
else:
    print("Goodput/throughput rows not found — check the tables above.")


### Post-hoc goodput from the raw export

You don't have to pick SLOs before the run: the per-request JSONL (Use Case 2 notebook) lets you compute goodput for **any** SLO combination afterwards. This is how this repo evaluates goodput today — the scenario scripts don't pass `--goodput`; SLOs are applied to the committed raw exports after the fact, so one run answers many SLO questions.

In [ ]:
jsonl_path = next((REPO_ROOT / OUTPUT_DIR).rglob("profile_export.jsonl"), None)
assert jsonl_path is not None, "No profile_export.jsonl found."

records = []
with open(jsonl_path) as f:
    for line in f:
        rec = json.loads(line)
        records.append({k: (v["value"] if isinstance(v, dict) and "value" in v else v)
                        for k, v in rec.get("metrics", {}).items()})
df = pd.DataFrame(records)


def goodput_pct(df, slos):
    """% of requests meeting every metric<=threshold pair in `slos` (ms)."""
    ok = pd.Series(True, index=df.index)
    for metric, threshold in slos.items():
        ok &= df[metric] <= threshold
    return 100 * ok.mean()


for name, slos in {
    "strict":   {"time_to_first_token": 250, "request_latency": 500},
    "balanced": {"time_to_first_token": TTFT_SLO, "request_latency": LATENCY_SLO},
    "relaxed":  {"time_to_first_token": 600, "request_latency": 2500},
}.items():
    print(f"{name:9s} {slos} -> {goodput_pct(df, slos):5.1f}% of requests pass")


In [ ]:
import matplotlib.pyplot as plt

# How sensitive is goodput to the TTFT threshold (latency SLO held fixed)?
thresholds = range(100, 1501, 50)
pcts = [goodput_pct(df, {"time_to_first_token": t, "request_latency": LATENCY_SLO})
        for t in thresholds]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(thresholds), pcts, marker=".")
ax.axvline(TTFT_SLO, linestyle="--", alpha=0.5)
ax.set_xlabel("TTFT SLO (ms)")
ax.set_ylabel(f"goodput % (latency SLO fixed at {LATENCY_SLO} ms)")
ax.set_title("Goodput sensitivity to the TTFT threshold")
plt.tight_layout()


### Notes

- A steep sensitivity curve means many requests sit just past the threshold — small provisioning or SLO changes swing goodput a lot; a flat curve means the SLO isn't the binding constraint.
- Use goodput to compare *deployments or models at equal cost*: two endpoints with identical raw throughput can differ hugely in the fraction of users actually served within SLO.